In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error, median_absolute_error, max_error
import numpy as np

In [ ]:
val_test = pd.read_csv("/kaggle/input/ccc2025/val_test.csv")
data = pd.read_csv("/kaggle/input/ccc2025/data.csv")
train = pd.read_csv("/kaggle/input/ccc2025/train.csv")

new_train = train.drop_duplicates().merge(data[["image","group_id"]], on="group_id", how='left')
new_train = new_train.sort_values("group_id")
new_train.index = train.sort_values("group_id").index
new_train = new_train.sort_index()
new_val_test = val_test.drop_duplicates().merge(data[["image","group_id"]], on="group_id", how='left')
new_val_test = new_val_test.sort_values("group_id")
new_val_test.index = val_test.sort_values("group_id").index
new_val_test = new_val_test.sort_index()

In [ ]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
import os
# 1. Загрузка предобученной модели (например, ResNet50)
model = models.resnet50(pretrained=True)
# Удаляем последний слой (classification head)
model = torch.nn.Sequential(*list(model.children())[:-1])
model.eval()  # Переводим модель в режим оценки

# 2. Предобработка изображений
transform = transforms.Compose([
    transforms.Resize((512,512)),
    transforms.CenterCrop(448),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 3. Функция для извлечения признаков из изображения
def extract_features(image_path):
    try:
        image = Image.open(image_path).convert('RGB')
        image = transform(image).unsqueeze(0)  # Добавляем размерность батча
        with torch.no_grad():
            features = model(image)
        return features.flatten().numpy()  # Преобразуем в numpy array
    except Exception as e:
        print(f"Error processing image {image_path}: {e}")
        return None  # Или вектор нулей, если ошибка

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 194MB/s]


In [ ]:
path = "/kaggle/input/ccc2025/data/images/"
# Путь к папке с изображениями
image_folder = "/kaggle/input/ccc2025/data/images/" # Указать правильный путь

# 5. Извлечение признаков для каждого изображения и создание признакового набора
features = []
targets = []
image_names = [] # Для отладки

for index, row in new_train.iterrows():
    image_name = row['image']
    image_path = os.path.join(image_folder, image_name)
    feature_vector = extract_features(image_path)

    if index % 1000 == 0 :
        print(index)
    if feature_vector is not None:
        features.append(feature_vector)
        targets.append(row['percentage_of_readiness'])
        image_names.append(image_name) # Для отладки

features = np.array(features)
targets = np.array(targets)

0
1000
2000
3000
4000
5000
6000
7000
8000
9000


In [ ]:
# 5. Извлечение признаков для каждого изображения и создание признакового набора
features_new_val_test = []
image_names_new_val_test = [] # Для отладки

for index, row in new_val_test.iterrows():
    image_name = row['image']
    image_path = os.path.join(image_folder, image_name)
    feature_vector = extract_features(image_path)

    if index % 1000 == 0 :
        print(index)
    if feature_vector is not None:
        features_new_val_test.append(feature_vector)
        #targets.append(row['percentage_of_readiness'])
        image_names_new_val_test.append(image_name) # Для отладки

features_new_val_test = np.array(features_new_val_test)

0
1000
2000
3000


In [ ]:
import lightgbm as lgb
X_train, X_test, y_train, y_test = train_test_split(features, targets, test_size=0.2, random_state=42)

# Параметры для LightGBM
params = {
    'objective': 'mae',  # Mean Absolute Error
    'metric': 'mae',
    'n_estimators': 427,   # Количество деревьев
    'learning_rate': 0.05,  # Скорость обучения
    'feature_fraction': 0.9, # Доля признаков для каждого дерева
    'bagging_fraction': 0.8,  # Доля данных для каждого дерева
    'bagging_freq': 1,# Частота бэггинга
    'reg_alpha': 0.01,       # L1 регуляризация
    'reg_lambda': 0.01,      # L2 регуляризация
    'min_child_samples': 20, # Минимальное кол-во samples в листе
    #'verbose': -1,           # Уровень verbosity (отключаем логи)
    'random_state': 42,
    'n_jobs': -1,           # Используем все доступные ядра
    'seed': 42
}



# Создание набора данных LightGBM
train_data_lgb = lgb.Dataset(X_train, label=y_train)
val_data_lgb = lgb.Dataset(X_test, label=y_test, reference=train_data_lgb)

# Обучение модели LightGBM
model = lgb.train(params,
                    train_data_lgb,
                    num_boost_round=100, # Количество итераций обучения (можно увеличить)
                    valid_sets=[train_data_lgb, val_data_lgb]
                 )

# 6. Предсказание
predictions = model.predict(X_test, num_iteration=model.best_iteration)
predictions = np.clip(predictions, 0, 100)  # Ensure predictions are within the valid range

# 7. Вычисление MAE
mae = mean_absolute_error(y_test, predictions)
print(f'Mean Absolute Error (MAE): {mae}')
# Вычисляем метрики
mse = mean_squared_error(y_test, predictions)
rmse = np.sqrt(mse)  # RMSE вычисляется вручную
r2 = r2_score(y_test, predictions)
mape = mean_absolute_percentage_error(y_test, predictions)
medae = median_absolute_error(y_test, predictions)
maxe = max_error(y_test, predictions)


# Выводим результаты
print(f"MAE: {mae}")
print(f"MSE: {mse}")
print(f"RMSE: {rmse}")
print(f"R-squared: {r2}")
print(f"MAPE: {mape}")
print(f"Median Absolute Error: {medae}")
print(f"Max Error: {maxe}")

/usr/local/lib/python3.10/dist-packages/lightgbm/engine.py:204: UserWarning: Found `n_estimators` in params. Will use it instead of argument
  _log_warning(f"Found `{alias}` in params. Will use it instead of argument")


[LightGBM] [Warning] seed is set=42, random_state=42 will be ignored. Current value: seed=42
[LightGBM] [Warning] seed is set=42, random_state=42 will be ignored. Current value: seed=42
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.172296 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 522237
[LightGBM] [Info] Number of data points in the train set: 7253, number of used features: 2048
[LightGBM] [Warning] seed is set=42, random_state=42 will be ignored. Current value: seed=42
[LightGBM] [Info] Start training from score 50.000000
Mean Absolute Error (MAE): 12.058760000147448
MAE: 12.058760000147448
MSE: 276.24602910780663
RMSE: 16.62065068244341
R-squared: 0.6859096061023895
MAPE: 769060622046685.6
Median Absolute Error: 8.679021217651968
Max Error: 65.3705749399385


In [ ]:
# 6. Предсказание
predictions = model.predict(features_new_val_test, num_iteration=model.best_iteration)
predictions = np.clip(predictions, 0, 100)  # Ensure predictions are within the valid range


test_df = pd.read_csv("/kaggle/input/ccc2025/val_test.csv")
test_df["percentage_of_readiness"] = predictions
test_df.to_csv("test_df.csv", index=False)